# Reinforcement Learning — Học tăng cường

## 1. Đặt vấn đề

Đến giờ, các bài lab đều thuộc loại **học có giám sát**: cho input + label, học hàm $f(x) \to y$. Nhưng nhiều bài toán không có "đáp án đúng" sẵn:
- Đi tới đâu trong bàn cờ vây để thắng?
- Robot nên đẩy chân nào để tiến về đích?
- Xe tự lái nên rẽ trái hay đi thẳng?

Ở những bài này, ta chỉ biết: "sau hàng loạt hành động, nếu đạt mục tiêu thì có *thưởng* (reward), nếu thất bại thì *phạt*". Đây là lúc **Reinforcement Learning** (RL) lên ngôi.

### Cơ chế cơ bản

Một agent (tác tử) tương tác với một môi trường (environment) qua các bước thời gian:

1. Tại bước $t$, môi trường ở **state** $s_t$.
2. Agent chọn **action** $a_t$ theo chính sách $\pi(a | s)$.
3. Môi trường chuyển sang state mới $s_{t+1}$ và trả **reward** $r_t$.
4. Lặp lại cho đến khi kết thúc episode.

Mục tiêu: tìm chính sách $\pi$ tối đa hoá **return tích luỹ**:
$$G_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k}$$

với $\gamma \in [0, 1)$ là **discount factor** — quan tâm đến reward gần hiện tại hơn reward xa tương lai.

## 2. Các khái niệm trung tâm

### Value function
$V^\pi(s) = \mathbb{E}_\pi[G_t \mid s_t = s]$ — kỳ vọng tổng reward nếu bắt đầu ở state $s$ và đi theo chính sách $\pi$.

### Q-function (action-value)
$Q^\pi(s, a) = \mathbb{E}_\pi[G_t \mid s_t = s, a_t = a]$ — kỳ vọng tổng reward nếu ở state $s$ chọn action $a$ rồi mới đi theo $\pi$.

### Phương trình Bellman
$$Q^\pi(s, a) = r(s, a) + \gamma \sum_{s'} P(s' | s, a) \sum_{a'} \pi(a' | s') Q^\pi(s', a')$$

Đây là phương trình hồi quy: giá trị Q hiện tại = reward ngay + discount × giá trị Q tương lai.

Ta sẽ tập trung vào **Q-learning** — một thuật toán RL không cần biết $P(s'|s,a)$, chỉ cần tương tác với môi trường.

### Sơ đồ vòng lặp Agent ↔ Environment


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis('off')

# Agent box
agent = FancyBboxPatch((1, 3.5), 2.5, 1.5, boxstyle='round,pad=0.1',
                       facecolor='#cce5ff', edgecolor='black', linewidth=1.5)
ax.add_patch(agent)
ax.text(2.25, 4.25, 'Agent', ha='center', va='center', fontsize=14, fontweight='bold')

# Environment box
env = FancyBboxPatch((6, 3.5), 2.5, 1.5, boxstyle='round,pad=0.1',
                     facecolor='#fff3cd', edgecolor='black', linewidth=1.5)
ax.add_patch(env)
ax.text(7.25, 4.25, 'Environment', ha='center', va='center', fontsize=14, fontweight='bold')

# Action arrow (Agent -> Env)
ax.add_patch(FancyArrowPatch((3.5, 4.6), (6, 4.6), arrowstyle='->',
                             mutation_scale=20, color='black', linewidth=2))
ax.text(4.75, 4.85, 'action $a_t$', ha='center', fontsize=12, color='darkred')

# State + Reward arrow (Env -> Agent)
ax.add_patch(FancyArrowPatch((6, 3.9), (3.5, 3.9), arrowstyle='->',
                             mutation_scale=20, color='black', linewidth=2))
ax.text(4.75, 3.55, 'state $s_{t+1}$,  reward $r_t$', ha='center', fontsize=12, color='darkgreen')

# Cycle indicator
ax.text(4.75, 2.5, '↻  Lặp lại đến hết episode', ha='center', fontsize=11, style='italic')

# Goal box at bottom
ax.text(5, 1.2, 'Mục tiêu: chọn $\\pi(a|s)$ tối đa hoá $G_t = \\sum_{k=0}^{\\infty}\\gamma^k r_{t+k}$',
        ha='center', fontsize=12, bbox=dict(boxstyle='round', facecolor='lightgray'))

ax.set_title('Reinforcement Learning loop: Agent ↔ Environment', fontsize=14)
plt.tight_layout(); plt.show()


## 3. Q-Learning

Ý tưởng: học một bảng $Q[s, a]$ chứa giá trị Q ước lượng. Cập nhật bằng công thức:
$$
Q[s, a] \leftarrow Q[s, a] + \alpha \big(r + \gamma \max_{a'} Q[s', a'] - Q[s, a]\big)
$$

Trong đó:
- $\alpha$ là learning rate.
- $r + \gamma \max_{a'} Q[s', a']$ là **target** — giá trị Q nên bằng nếu agent chọn tối ưu ở bước kế.
- Hiệu giữa target và Q hiện tại gọi là **TD error** (temporal difference).

### Khám phá vs khai thác (Exploration vs Exploitation)

Nếu agent luôn chọn action có Q lớn nhất → "khai thác" (exploitation), không khám phá hành động mới.
Nếu chọn ngẫu nhiên → "khám phá" (exploration), không tận dụng kiến thức đã học.

Cân bằng: **ε-greedy** — với xác suất $\epsilon$ chọn ngẫu nhiên, ngược lại chọn action có Q max. Thường $\epsilon$ giảm dần theo thời gian (lúc đầu khám phá nhiều, sau ít dần).

### Trực quan: ε-greedy và discount factor

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Minh hoạ ε-greedy — vì sao cần exploration.
np.random.seed(0)
n_arms = 5
true_means = np.array([0.1, 0.3, 0.5, 0.7, 0.4])  # 5 cánh tay
n_steps = 200

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, eps, color in [('ε=0 (greedy)', 0.0, 'red'),
                          ('ε=0.1', 0.1, 'blue'),
                          ('ε=1.0 (random)', 1.0, 'gray')]:
    np.random.seed(0)
    Q = np.zeros(n_arms); n = np.zeros(n_arms); rewards = []
    for t in range(n_steps):
        if np.random.rand() < eps:
            a = np.random.randint(n_arms)
        else:
            a = int(np.argmax(Q))
        r = np.random.randn() * 0.1 + true_means[a]
        n[a] += 1
        Q[a] += (r - Q[a]) / n[a]
        rewards.append(r)
    cum = np.cumsum(rewards) / np.arange(1, n_steps + 1)
    axes[0].plot(cum, label=label, color=color)

axes[0].axhline(true_means.max(), color='black', linestyle='--', alpha=0.5, label='Optimal')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Reward trung bình')
axes[0].set_title('ε-greedy: cân bằng Exploration vs Exploitation')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Discount factor
gammas = [0.5, 0.9, 0.99]
ts = np.arange(50)
for g in gammas:
    axes[1].plot(ts, g**ts, label=f'γ={g}')
axes[1].set_xlabel('Bước thời gian k'); axes[1].set_ylabel('$\\gamma^k$')
axes[1].set_title('Discount factor: reward càng xa càng được \"giảm giá\"')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()


# THỰC HÀNH 1: Q-Learning trên FrozenLake

**FrozenLake** là môi trường đơn giản trong gym:
- Lưới 4×4. Agent bắt đầu ở góc trái trên, mục tiêu là góc phải dưới.
- Có một số ô "hố" — rơi vào là kết thúc episode, reward 0.
- Đến đích → reward 1.
- Hai biến thể: `is_slippery=True` (đi trượt — khó), `False` (đi đúng hướng — dễ). Ta dùng phiên bản dễ trước.

Cài thư viện trước (chỉ chạy 1 lần):
```bash
pip install gymnasium
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

try:
    import gymnasium as gym
except ImportError:
    print('Chưa cài gymnasium — chạy: pip install gymnasium')
    raise

np.random.seed(42)
random.seed(42)

env = gym.make('FrozenLake-v1', is_slippery=False)
n_states  = env.observation_space.n      # 16
n_actions = env.action_space.n           # 4 (trái, dưới, phải, lên)
print(f'Số state: {n_states}, số action: {n_actions}')

# Bản đồ FrozenLake mặc định:
# S F F F
# F H F H
# F F F H
# H F F G
# S = start, F = frozen (đi được), H = hole, G = goal.

In [ ]:
Q = np.zeros((n_states, n_actions))

# Hyperparams
alpha    = 0.8       # learning rate
gamma    = 0.95      # discount factor
epsilon  = 1.0       # exploration rate (giảm dần)
eps_decay = 0.995
eps_min  = 0.01
n_episodes = 2000

rewards_history = []

for ep in range(n_episodes):
    state, _ = env.reset()
    total_reward = 0
    done = False
    while not done:
        # ε-greedy: chọn ngẫu nhiên với xác suất ε.
        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = int(np.argmax(Q[state]))

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # Cập nhật Q-table.
        Q[state, action] += alpha * (reward + gamma * np.max(Q[next_state]) - Q[state, action])

        state = next_state
        total_reward += reward

    epsilon = max(eps_min, epsilon * eps_decay)
    rewards_history.append(total_reward)

    if (ep + 1) % 200 == 0:
        avg = np.mean(rewards_history[-100:])
        print(f'Episode {ep+1:4d}  avg reward last 100: {avg:.2f}  ε = {epsilon:.3f}')

In [ ]:
# Vẽ tỷ lệ thành công theo episode (rolling window 100).
rewards_arr = np.array(rewards_history)
rolling = np.convolve(rewards_arr, np.ones(100)/100, mode='valid')

plt.figure(figsize=(10, 4))
plt.plot(rolling)
plt.xlabel('Episode'); plt.ylabel('Tỷ lệ thắng (rolling 100)')
plt.title('Q-Learning trên FrozenLake (deterministic)')
plt.grid(alpha=0.3); plt.show()

In [ ]:
# Kiểm tra chính sách đã học bằng cách chạy 100 episode greedy.
wins = 0
for _ in range(100):
    state, _ = env.reset()
    done = False
    while not done:
        action = int(np.argmax(Q[state]))
        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        if reward > 0: wins += 1
print(f'Tỷ lệ thắng greedy: {wins}/100')

# In bảng Q-table cho 4 ô đầu tiên
print('\nQ-table (4 hàng đầu — state 0,1,2,3):')
print('Action: [trái, dưới, phải, lên]')
for s in range(4):
    print(f'State {s}: {Q[s].round(3)}')

### Trực quan chính sách

Vẽ mũi tên trên bản đồ 4×4 thể hiện action mà agent chọn ở mỗi state.

In [ ]:
arrows = ['←', '↓', '→', '↑']
lake_map = ['S', 'F', 'F', 'F',
            'F', 'H', 'F', 'H',
            'F', 'F', 'F', 'H',
            'H', 'F', 'F', 'G']

fig, ax = plt.subplots(figsize=(5, 5))
for i in range(4):
    for j in range(4):
        s = i * 4 + j
        cell = lake_map[s]
        if cell in ('S', 'G'):
            color = 'lightgreen' if cell == 'G' else 'lightyellow'
        elif cell == 'H':
            color = 'lightcoral'
        else:
            color = 'lightblue'
        ax.add_patch(plt.Rectangle((j, 3-i), 1, 1, facecolor=color, edgecolor='black'))
        if cell == 'F' or cell == 'S':
            best_a = int(np.argmax(Q[s]))
            ax.text(j + 0.5, 3-i + 0.5, arrows[best_a], ha='center', va='center', fontsize=20)
        else:
            ax.text(j + 0.5, 3-i + 0.5, cell, ha='center', va='center', fontsize=18, fontweight='bold')
ax.set_xlim(0, 4); ax.set_ylim(0, 4); ax.set_xticks([]); ax.set_yticks([])
ax.set_title('Chính sách đã học (mũi tên = action greedy)')
plt.show()

# THỰC HÀNH 2: Deep Q-Network (DQN) trên CartPole

Q-table chỉ làm được khi state space hữu hạn và nhỏ. Với môi trường có state liên tục (CartPole: vị trí, vận tốc, góc, vận tốc góc), ta dùng một mạng nơ-ron để xấp xỉ Q-function. Đây là **DQN** — bài báo nổi tiếng của DeepMind 2013.

**CartPole**: cây sào dựng trên xe trượt. Agent đẩy xe trái/phải để giữ sào không ngã. Mỗi bước sào còn đứng → +1 reward, ngã → kết thúc.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

env = gym.make('CartPole-v1')
state_dim = env.observation_space.shape[0]   # 4
action_dim = env.action_space.n              # 2
print('state_dim:', state_dim, ' action_dim:', action_dim)

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.ReLU(),
            nn.Linear(hidden, action_dim),
        )
    def forward(self, s):
        return self.net(s)

policy_net = QNetwork(state_dim, action_dim).to(device)
# Target network: bản sao tách riêng, cập nhật chậm để training ổn định.
target_net = QNetwork(state_dim, action_dim).to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# Replay buffer: lưu các transition (s, a, r, s', done) để sample mini-batch.
# Phá vỡ tương quan thời gian — quan trọng để training ổn định.
replay = deque(maxlen=10000)
batch_size = 64
gamma = 0.99
epsilon = 1.0
eps_decay = 0.995
eps_min = 0.05
target_update_freq = 10   # update target_net mỗi 10 episode

In [ ]:
def select_action(state, eps):
    if random.random() < eps:
        return env.action_space.sample()
    with torch.no_grad():
        s = torch.FloatTensor(state).unsqueeze(0).to(device)
        return int(policy_net(s).argmax(dim=1).item())

def train_step():
    if len(replay) < batch_size:
        return
    batch = random.sample(replay, batch_size)
    states, actions, rewards, next_states, dones = zip(*batch)
    states      = torch.FloatTensor(np.array(states)).to(device)
    actions     = torch.LongTensor(actions).unsqueeze(1).to(device)
    rewards     = torch.FloatTensor(rewards).to(device)
    next_states = torch.FloatTensor(np.array(next_states)).to(device)
    dones       = torch.FloatTensor(dones).to(device)

    # Q-value hiện tại với action đã chọn.
    q_pred = policy_net(states).gather(1, actions).squeeze(1)
    # Target: r + γ max_a' Q_target(s', a'), với done=1 thì không cộng phần future.
    with torch.no_grad():
        q_next = target_net(next_states).max(1)[0]
        q_target = rewards + gamma * q_next * (1 - dones)

    loss = criterion(q_pred, q_target)
    optimizer.zero_grad(); loss.backward()
    # Clip gradient để tránh exploding.
    torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 10.0)
    optimizer.step()
    return loss.item()

In [ ]:
n_episodes = 300
ep_rewards = []

for ep in range(n_episodes):
    state, _ = env.reset(seed=ep)
    total = 0
    done = False
    while not done:
        action = select_action(state, epsilon)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        replay.append((state, action, reward, next_state, float(done)))
        train_step()
        state = next_state
        total += reward

    epsilon = max(eps_min, epsilon * eps_decay)
    ep_rewards.append(total)

    if ep % target_update_freq == 0:
        target_net.load_state_dict(policy_net.state_dict())

    if (ep + 1) % 30 == 0:
        avg = np.mean(ep_rewards[-30:])
        print(f'Ep {ep+1:3d}  avg reward last 30: {avg:6.2f}  ε = {epsilon:.3f}')

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(ep_rewards, alpha=0.4, label='Reward mỗi episode')
if len(ep_rewards) >= 30:
    rolling = np.convolve(ep_rewards, np.ones(30)/30, mode='valid')
    plt.plot(range(29, len(ep_rewards)), rolling, label='Trung bình 30 episode', linewidth=2)
plt.xlabel('Episode'); plt.ylabel('Total reward')
plt.title('DQN trên CartPole-v1 (max reward = 500)')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## Tổng kết

1. **RL** giải bài toán "agent học hành động qua thử-sai-thưởng-phạt" — không cần label cho từng bước.
2. **Q-Learning** học bảng $Q[s,a]$ bằng phương trình Bellman + ε-greedy. Tốt cho state space nhỏ rời rạc.
3. **DQN** thay bảng Q bằng mạng nơ-ron — xử lý được state liên tục. Hai trick quan trọng:
   - **Replay buffer**: phá tương quan thời gian giữa các transition.
   - **Target network**: bản sao Q-network cập nhật chậm, giúp target không nhảy lung tung.
4. Sau DQN còn nhiều thuật toán nâng cao: Double DQN, Dueling DQN, Policy Gradient, A2C, PPO, SAC... đó là các bước tiếp theo nếu em muốn đi sâu vào RL.

# BÀI TẬP VỀ NHÀ

## Bài 1: FrozenLake với is_slippery=True

Train Q-Learning trên `FrozenLake-v1` với `is_slippery=True` (mỗi action có 1/3 xác suất đi đúng hướng, 2/3 trượt sang bên). Bài khó hơn nhiều.

**Yêu cầu:**
1. Chạy với 5000 episode (vì khó hơn cần nhiều thử-sai hơn).
2. Vẽ rolling success rate.
3. So sánh với phiên bản deterministic ở lab — agent đạt tỷ lệ thắng cao nhất là bao nhiêu?
4. Thử thay đổi `alpha` (0.1, 0.5, 0.9), `gamma` (0.9, 0.99) — báo cáo ảnh hưởng.

*Gợi ý:* với is_slippery=True, agent không thể luôn đến đích bằng 1 đường thẳng — phải học chính sách "an toàn" hơn (tránh ô gần hố).

## Bài 2: Cải thiện DQN trên CartPole

Bài lab DQN có thể chưa hội tụ thật tốt trong 300 episode. Hãy:
1. Train 1000 episode. Đến khi nào agent đạt reward 500 (tối đa) đều đặn?
2. Thử các kỹ thuật nâng cao:
   - **Double DQN**: thay `target_net(next_states).max(1)[0]` bằng `target_net(next_states).gather(1, policy_net(next_states).argmax(1, keepdim=True)).squeeze()`. Giảm overestimation bias của Q.
   - **Soft update**: thay vì copy hard mỗi 10 episode, dùng `target_param.data.copy_(τ * policy_param.data + (1-τ) * target_param.data)` với τ = 0.005 mỗi step.
3. Báo cáo: hai kỹ thuật này có giúp hội tụ nhanh hơn không?

## Bài 3: Render và quay video (nâng cao)

Sau khi train xong, render lại một episode bằng tay đã học và lưu thành file MP4.

```python
env_render = gym.make('CartPole-v1', render_mode='rgb_array')
from gymnasium.wrappers import RecordVideo
env_render = RecordVideo(env_render, video_folder='./videos', episode_trigger=lambda x: True)
state, _ = env_render.reset()
done = False
while not done:
    action = select_action(state, eps=0)   # greedy
    state, _, term, trunc, _ = env_render.step(action)
    done = term or trunc
env_render.close()
```

Cần `pip install moviepy`. Nộp video kết quả.

## Bài 4: Đọc thêm

Đọc paper *"Playing Atari with Deep Reinforcement Learning"* (DeepMind, 2013) — bản gốc DQN. Trả lời ngắn:
1. Vì sao DeepMind chọn Atari để demo (chứ không phải robot thật)?
2. Ý nghĩa của **frame stacking** (nối 4 frame liên tiếp làm input)?
3. Kiến trúc mạng dùng cho ảnh Atari là gì? (gợi ý: liên quan tới CNN)

## Hạn nộp
30/04/2026.